# ============================================================
# Telecom Customer Analytics Platform
# Generalized Data Profiling Notebook — All Datasets
# ============================================================
# Datasets:
#   1. pk.csv              — Pakistan Cities (Geographic Reference)
#   2. WA_Fn-UseC_-Telco-Customer-Churn.csv — IBM Telco Churn (Original)
#   3. Telco_customer_churn.xlsx            — IBM Telco Churn (Extended)
#   4. 410.csv             — OpenCelliD Cell Tower Data (Pakistan MCC=410)
# ============================================================

## Phase 1 — Import Libraries & Global Configuration

In [20]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Output Folders ──────────────────────────────────────────
OUTPUT_DIR  = Path('profiling_outputs')
PLOTS_DIR   = OUTPUT_DIR / 'plots'
RESULTS_DIR = OUTPUT_DIR / 'results'

for d in [OUTPUT_DIR, PLOTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Color Palette ────────────────────────────────────────────
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0',
           '#FF9800', '#00BCD4', '#E91E63', '#607D8B']

print('Libraries loaded. Outputs will be saved to:', OUTPUT_DIR.resolve())

Libraries loaded. Outputs will be saved to: D:\telecom-customer-analytics-platform\notebooks\profiling_outputs


## Phase 2 — Generalized Profiling Engine

In [21]:
# ============================================================
# GENERALIZED PROFILING FUNCTIONS
# ============================================================

# ── Master results store (accumulates across all datasets) ───
MASTER_RESULTS = {}


# ── 1. Dataset Overview ──────────────────────────────────────
def overview(df, name):
    """Rows, columns, memory usage."""
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    info = {
        'rows':    df.shape[0],
        'columns': df.shape[1],
        'memory_mb': round(mem_mb, 4)
    }
    print(f"\n{'='*60}")
    print(f"  DATASET OVERVIEW — {name}")
    print(f"{'='*60}")
    print(f"  Rows               : {info['rows']:,}")
    print(f"  Columns            : {info['columns']}")
    print(f"  Memory Usage (MB)  : {info['memory_mb']}")
    return info


# ── 2. Column Names ──────────────────────────────────────────
def column_names(df, name):
    """Print and return all column names."""
    print(f"\n{'='*60}")
    print(f"  COLUMN NAMES — {name}")
    print(f"{'='*60}")
    cols = list(df.columns)
    for i, c in enumerate(cols, 1):
        print(f"  {i:3}. {c}")
    return cols


# ── 3. Data Types ────────────────────────────────────────────
def data_types(df, name):
    """Return a DataFrame of column → dtype."""
    dtype_df = pd.DataFrame({
        'Column': df.columns,
        'DType':  df.dtypes.astype(str).values
    })
    print(f"\n{'='*60}")
    print(f"  DATA TYPES — {name}")
    print(f"{'='*60}")
    print(dtype_df.to_string(index=False))
    return dtype_df


# ── 4. Summary Statistics ────────────────────────────────────
def summary_statistics(df, name):
    """describe() on numeric + object columns."""
    print(f"\n{'='*60}")
    print(f"  SUMMARY STATISTICS — {name}")
    print(f"{'='*60}")
    numeric_desc = df.describe(include=[np.number])
    object_desc  = df.describe(include=['object'])
    print("\n[Numeric]")
    print(numeric_desc)
    if not object_desc.empty:
        print("\n[Categorical]")
        print(object_desc)
    return numeric_desc, object_desc


# ── 5. Duplicate Rows ────────────────────────────────────────
def duplicate_analysis(df, name, id_col=None):
    """Count duplicate rows and optionally duplicate ID values."""
    total_dups = df.duplicated().sum()
    dup_pct    = round(total_dups / len(df) * 100, 2)
    result = {'total_duplicate_rows': int(total_dups),
              'duplicate_pct': dup_pct}
    print(f"\n{'='*60}")
    print(f"  DUPLICATE ANALYSIS — {name}")
    print(f"{'='*60}")
    print(f"  Duplicate Rows       : {total_dups}")
    print(f"  Duplicate Percentage : {dup_pct:.2f}%")
    if id_col and id_col in df.columns:
        id_dups = df[id_col].duplicated().sum()
        result['duplicate_id_col'] = id_col
        result['duplicate_id_count'] = int(id_dups)
        print(f"  Duplicate '{id_col}'     : {id_dups}")
    return result


# ── 6. Missing Values ────────────────────────────────────────
def missing_values(df, name):
    """Count and percentage of NaN per column."""
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    mv_df = pd.DataFrame({
        'Missing_Count': missing,
        'Missing_Pct':   missing_pct
    }).sort_values('Missing_Count', ascending=False)
    print(f"\n{'='*60}")
    print(f"  MISSING VALUES — {name}")
    print(f"{'='*60}")
    with_missing = mv_df[mv_df['Missing_Count'] > 0]
    if with_missing.empty:
        print("  No missing values found.")
    else:
        print(with_missing.to_string())
    total_mv = int(missing.sum())
    print(f"\n  Total Missing Cells  : {total_mv}")
    print(f"  Total Cells          : {df.shape[0] * df.shape[1]}")
    return mv_df


# ── 7. Blank / Whitespace Values ─────────────────────────────
def blank_whitespace(df, name, extra_cols=None):
    """Detect empty strings in object + specified columns."""
    print(f"\n{'='*60}")
    print(f"  BLANK / WHITESPACE VALUES — {name}")
    print(f"{'='*60}")
    obj_cols = list(df.select_dtypes(include='object').columns)
    if extra_cols:
        obj_cols = list(set(obj_cols + extra_cols))
    blank_summary = {}
    for col in obj_cols:
        if col in df.columns:
            blank_count = df[col].astype(str).str.strip().eq('').sum()
            if blank_count > 0:
                blank_summary[col] = blank_count
                print(f"  {col:<30}: {blank_count} blank values")
    if not blank_summary:
        print("  No blank/whitespace values found.")
    return blank_summary


# ── 8. Unique Values ────────────────────────────────────────
def unique_values(df, name):
    """Count distinct values per column."""
    uv_df = pd.DataFrame({
        'Column':        df.columns,
        'Unique_Count':  df.nunique().values,
        'Total_Count':   len(df),
        'Unique_Pct':    (df.nunique() / len(df) * 100).round(2).values
    })
    print(f"\n{'='*60}")
    print(f"  UNIQUE VALUES — {name}")
    print(f"{'='*60}")
    print(uv_df.to_string(index=False))
    return uv_df


# ── 9. Cardinality ──────────────────────────────────────────
def cardinality(df, name, low_thresh=5, med_thresh=20):
    """Classify each column as Low / Medium / High cardinality."""
    def classify(n):
        if n <= low_thresh:  return 'Low'
        if n <= med_thresh:  return 'Medium'
        return 'High'
    card_df = pd.DataFrame({
        'Column':      df.columns,
        'Unique':      df.nunique().values,
        'Cardinality': [classify(n) for n in df.nunique().values]
    })
    print(f"\n{'='*60}")
    print(f"  CARDINALITY — {name}")
    print(f"{'='*60}")
    print(card_df.to_string(index=False))
    return card_df


# ── 10. Numeric Columns ─────────────────────────────────────
def numeric_columns(df, name):
    """Auto-detect and list numeric columns."""
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    print(f"\n{'='*60}")
    print(f"  NUMERIC COLUMNS — {name}")
    print(f"{'='*60}")
    for c in num_cols:
        print(f"  {c:<30} {str(df[c].dtype):<12}")
    print(f"\n  Total Numeric Columns : {len(num_cols)}")
    return num_cols


# ── 11. Outlier Detection (IQR + Boxplots) ──────────────────
def outlier_detection(df, name, num_cols=None, max_plots=6, color=None):
    """IQR-based outlier count + boxplots saved to disk."""
    if num_cols is None:
        num_cols = df.select_dtypes(include=np.number).columns.tolist()
    num_cols = [c for c in num_cols if c in df.columns]
    if not num_cols:
        print('  No numeric columns for outlier detection.')
        return {}

    print(f"\n{'='*60}")
    print(f"  OUTLIER DETECTION (IQR) — {name}")
    print(f"{'='*60}")

    outlier_info = {}
    for col in num_cols:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lo  = Q1 - 1.5 * IQR
        hi  = Q3 + 1.5 * IQR
        n_out = int(((df[col] < lo) | (df[col] > hi)).sum())
        outlier_info[col] = {'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
                             'lower_bound': lo, 'upper_bound': hi,
                             'outlier_count': n_out}
        print(f"  {col:<28} Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}  "
              f"Bounds=[{lo:.2f}, {hi:.2f}]  Outliers={n_out}")

    # Boxplots
    plot_cols = num_cols[:max_plots]
    ncols = min(3, len(plot_cols))
    nrows = int(np.ceil(len(plot_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()
    clr  = color or PALETTE[0]
    for i, col in enumerate(plot_cols):
        axes[i].boxplot(df[col].dropna(), patch_artist=True,
                        boxprops=dict(facecolor=clr, alpha=0.7))
        axes[i].set_title(col, fontsize=9)
        axes[i].set_xlabel('')
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    fig.suptitle(f'Boxplots — {name}', fontsize=12, y=1.01)
    plt.tight_layout()
    path = PLOTS_DIR / f"{name.replace(' ', '_')}_boxplots.png"
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  Boxplot saved → {path}")
    return outlier_info


# ── 12. Frequency Distribution ──────────────────────────────
def frequency_distribution(df, name, max_cats=10, color=None):
    """Value counts for categorical columns + bar chart."""
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    if not cat_cols:
        print('  No categorical columns found.')
        return {}
    print(f"\n{'='*60}")
    print(f"  FREQUENCY DISTRIBUTION — {name}")
    print(f"{'='*60}")
    freq_results = {}
    clr = color or PALETTE[1]
    for col in cat_cols:
        vc = df[col].value_counts().head(max_cats)
        freq_results[col] = vc.to_dict()
        print(f"\n  [{col}]")
        print(vc.to_string())

    # Bar chart for first categorical with ≤20 unique vals
    chart_cols = [c for c in cat_cols if df[c].nunique() <= 20]
    if chart_cols:
        col  = chart_cols[0]
        vc   = df[col].value_counts().head(15)
        fig, ax = plt.subplots(figsize=(8, 4))
        vc.plot(kind='bar', ax=ax, color=clr, alpha=0.85, edgecolor='white')
        ax.set_title(f'Frequency — {col} ({name})', fontsize=11)
        ax.set_xlabel(col)
        ax.set_ylabel('Count')
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.tight_layout()
        path = PLOTS_DIR / f"{name.replace(' ', '_')}_freq_{col}.png"
        plt.savefig(path, dpi=120, bbox_inches='tight')
        plt.close()
        print(f"  Bar chart saved → {path}")
    return freq_results


# ── 13. Correlation Analysis ─────────────────────────────────
def correlation_analysis(df, name, color='Blues'):
    """Pearson correlation matrix + heatmap for numeric columns."""
    num_df = df.select_dtypes(include=np.number)
    if num_df.shape[1] < 2:
        print(f"  Not enough numeric columns for correlation in {name}.")
        return None
    corr = num_df.corr()
    print(f"\n{'='*60}")
    print(f"  CORRELATION MATRIX — {name}")
    print(f"{'='*60}")
    print(corr.round(3).to_string())
    # Heatmap
    fig, ax = plt.subplots(figsize=(max(6, num_df.shape[1]),
                                    max(5, num_df.shape[1] - 1)))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap=color,
                linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title(f'Correlation Heatmap — {name}', fontsize=11)
    plt.tight_layout()
    path = PLOTS_DIR / f"{name.replace(' ', '_')}_correlation.png"
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  Heatmap saved → {path}")
    return corr


# ── 14. Memory Usage per Column ──────────────────────────────
def memory_usage(df, name):
    """Detailed memory usage per column."""
    mem = df.memory_usage(deep=True)
    mem_df = pd.DataFrame({
        'Column':   ['Index'] + list(df.columns),
        'Bytes':    mem.values,
        'KB':       (mem.values / 1024).round(3)
    }).sort_values('Bytes', ascending=False)
    total_kb = mem.sum() / 1024
    print(f"\n{'='*60}")
    print(f"  MEMORY USAGE — {name}")
    print(f"{'='*60}")
    print(mem_df.to_string(index=False))
    print(f"\n  TOTAL : {total_kb:.2f} KB  ({total_kb/1024:.4f} MB)")
    return mem_df


# ── Master runner: applies all 14 checks ─────────────────────
def run_full_profile(df, name, id_col=None,
                     force_numeric_cols=None,
                     blank_extra_cols=None,
                     dataset_color=None,
                     corr_cmap='Blues'):
    """
    Run all 14 profiling checks on df.
    Parameters
    ----------
    df                : pd.DataFrame
    name              : str  — display name for this dataset
    id_col            : str  — optional unique-ID column for duplicate check
    force_numeric_cols: list — override auto-detect for outlier/corr
    blank_extra_cols  : list — extra non-object cols to check for blanks
    dataset_color     : str  — hex color for charts
    corr_cmap         : str  — seaborn colormap for correlation heatmap
    """
    print('\n' + '█'*70)
    print(f'  PROFILING START : {name}')
    print('█'*70)

    result = {'name': name}

    result['overview']        = overview(df, name)
    result['columns']         = column_names(df, name)
    result['dtypes']          = data_types(df, name)
    result['stats']           = summary_statistics(df, name)
    result['duplicates']      = duplicate_analysis(df, name, id_col=id_col)
    result['missing']         = missing_values(df, name)
    result['blanks']          = blank_whitespace(df, name, extra_cols=blank_extra_cols)
    result['unique']          = unique_values(df, name)
    result['cardinality']     = cardinality(df, name)
    result['numeric_cols']    = numeric_columns(df, name)
    num_cols_for_analysis     = force_numeric_cols or result['numeric_cols']
    result['outliers']        = outlier_detection(df, name,
                                                  num_cols=num_cols_for_analysis,
                                                  color=dataset_color)
    result['frequency']       = frequency_distribution(df, name,
                                                       color=dataset_color)
    result['correlation']     = correlation_analysis(df, name, color=corr_cmap)
    result['memory']          = memory_usage(df, name)

    MASTER_RESULTS[name] = result
    print(f"\n  ✓ Profiling complete for [{name}]")
    print('█'*70 + '\n')
    return result


print('Profiling engine ready.')

Profiling engine ready.


## Phase 3 — Dataset 1: Pakistan Cities (pk.csv)

In [22]:
pk_df = pd.read_csv(r"D:\telecom-customer-analytics-platform\data\raw\pk.csv")

print("pk.csv loaded. Shape:", pk_df.shape)
pk_df.head()

pk.csv loaded. Shape: (146, 9)


,city,lat,lng,country,iso2,admin_name,capital,population,population_proper
0,Karachi,24.8600,67.0100,Pakistan,PK,Sindh,admin,14835000.0000,14835000.0000
1,Lahore,31.5497,74.3436,Pakistan,PK,Punjab,admin,11021000.0000,11021000.0000
2,Sialkot City,32.5000,74.5333,Pakistan,PK,Punjab,minor,3893672.0000,3893672.0000
3,Faisalabad,31.4180,73.0790,Pakistan,PK,Punjab,minor,3203846.0000,3203846.0000
4,Rawalpindi,33.6007,73.0679,Pakistan,PK,Punjab,minor,2098231.0000,2098231.0000


In [23]:
# ── Dataset-Specific Notes ────────────────────────────────────
# • Geographic reference data (146 Pakistani cities)
# • No customer ID column
# • 'capital' column has low cardinality (admin / minor / primary / NaN)
# • population & population_proper should be equal or near-equal
# • lat/lng are continuous geographic coordinates — outlier detection meaningful

pk_result = run_full_profile(
    df                = pk_df,
    name              = 'PK_Cities',
    id_col            = 'city',           # city acts as the identifier
    force_numeric_cols= ['lat', 'lng', 'population', 'population_proper'],
    blank_extra_cols  = ['admin_name', 'capital'],
    dataset_color     = PALETTE[0],        # blue
    corr_cmap         = 'YlOrRd'
)


██████████████████████████████████████████████████████████████████████
  PROFILING START : PK_Cities
██████████████████████████████████████████████████████████████████████

  DATASET OVERVIEW — PK_Cities
  Rows               : 146
  Columns            : 9
  Memory Usage (MB)  : 0.0489

  COLUMN NAMES — PK_Cities
    1. city
    2. lat
    3. lng
    4. country
    5. iso2
    6. admin_name
    7. capital
    8. population
    9. population_proper

  DATA TYPES — PK_Cities
           Column   DType
             city  object
              lat float64
              lng float64
          country  object
             iso2  object
       admin_name  object
          capital  object
       population float64
population_proper float64

  SUMMARY STATISTICS — PK_Cities

[Numeric]
           lat      lng    population  population_proper
count 146.0000 146.0000       88.0000            88.0000
mean   30.9300  71.0196   646726.5114        645972.4659
std     2.9526   2.7251  2016112.5185       20

In [24]:
# ── Dataset-Specific: Population Discrepancy Check ───────────
print('='*60)
print('  POPULATION vs POPULATION_PROPER CHECK — PK_Cities')
print('='*60)
mismatch = pk_df[pk_df['population'] != pk_df['population_proper']]
print(f'  Mismatches : {len(mismatch)}')
if len(mismatch) > 0:
    print(mismatch[['city', 'population', 'population_proper']].to_string(index=False))

# ── Capital Type Breakdown ────────────────────────────────────
print('\n  Capital Type Distribution:')
print(pk_df['capital'].value_counts(dropna=False))

  POPULATION vs POPULATION_PROPER CHECK — PK_Cities
  Mismatches : 60
               city  population  population_proper
               Zhob  88356.0000         50537.0000
             Gwadar  51901.0000         23364.0000
     Hyderabad City         NaN                NaN
            Narowal         NaN                NaN
     Khairpur Mir’s         NaN                NaN
           Khanewal         NaN                NaN
             Jhelum         NaN                NaN
            Haripur         NaN                NaN
          Shikarpur         NaN                NaN
         Rawala Kot         NaN                NaN
          Hafizabad         NaN                NaN
            Lodhran         NaN                NaN
           Malakand         NaN                NaN
        Attock City         NaN                NaN
            Batgram         NaN                NaN
            Matiari         NaN                NaN
             Ghotki         NaN                NaN
    Naushahr

## Phase 4 — Dataset 2: IBM Telco Churn Original (WA_Fn CSV)

In [25]:
# ── Load ─────────────────────────────────────────────────────
telco_csv_df = pd.read_csv(r"D:\telecom-customer-analytics-platform\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Phase 3 — Dataset 1: Pakistan Cities (pk.csv)
pk_df = pd.read_csv(r"D:\telecom-customer-analytics-platform\data\raw\pk.csv")

print("pk.csv loaded. Shape:", pk_df.shape)
pk_df.head()
print('WA_Fn CSV loaded. Shape:', telco_csv_df.shape)
telco_csv_df.head()

pk.csv loaded. Shape: (146, 9)
WA_Fn CSV loaded. Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.8500,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.9500,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.8500,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3000,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7000,151.65,Yes


In [26]:
# ── Dataset-Specific Notes ────────────────────────────────────
# • Classic IBM Telco dataset (7043 rows, 21 columns)
# • TotalCharges is numeric but may contain blank strings → converted
# • SeniorCitizen is 0/1 integer (binary flag, not really numeric)
# • customerID is high-cardinality identifier

# Pre-processing: fix TotalCharges before profiling
telco_csv_df['TotalCharges'] = pd.to_numeric(
    telco_csv_df['TotalCharges'].astype(str).str.strip(),
    errors='coerce'
)
blank_tc = telco_csv_df['TotalCharges'].isna().sum()
print(f'TotalCharges blanks coerced to NaN : {blank_tc}')

telco_csv_result = run_full_profile(
    df                 = telco_csv_df,
    name               = 'Telco_CSV',
    id_col             = 'customerID',
    force_numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges'],
    blank_extra_cols   = ['TotalCharges'],
    dataset_color      = PALETTE[2],    # green
    corr_cmap          = 'Greens'
)

TotalCharges blanks coerced to NaN : 11

██████████████████████████████████████████████████████████████████████
  PROFILING START : Telco_CSV
██████████████████████████████████████████████████████████████████████

  DATASET OVERVIEW — Telco_CSV
  Rows               : 7,043
  Columns            : 21
  Memory Usage (MB)  : 7.4196

  COLUMN NAMES — Telco_CSV
    1. customerID
    2. gender
    3. SeniorCitizen
    4. Partner
    5. Dependents
    6. tenure
    7. PhoneService
    8. MultipleLines
    9. InternetService
   10. OnlineSecurity
   11. OnlineBackup
   12. DeviceProtection
   13. TechSupport
   14. StreamingTV
   15. StreamingMovies
   16. Contract
   17. PaperlessBilling
   18. PaymentMethod
   19. MonthlyCharges
   20. TotalCharges
   21. Churn

  DATA TYPES — Telco_CSV
          Column   DType
      customerID  object
          gender  object
   SeniorCitizen   int64
         Partner  object
      Dependents  object
          tenure   int64
    PhoneService  object
   Multip


[Numeric]
       SeniorCitizen    tenure  MonthlyCharges  TotalCharges
count      7043.0000 7043.0000       7043.0000     7032.0000
mean          0.1621   32.3711         64.7617     2283.3004
std           0.3686   24.5595         30.0900     2266.7714
min           0.0000    0.0000         18.2500       18.8000
25%           0.0000    9.0000         35.5000      401.4500
50%           0.0000   29.0000         70.3500     1397.4750
75%           0.0000   55.0000         89.8500     3794.7375
max           1.0000   72.0000        118.7500     8684.8000

[Categorical]
        customerID gender Partner Dependents PhoneService MultipleLines InternetService OnlineSecurity OnlineBackup  \
count         7043   7043    7043       7043         7043          7043            7043           7043         7043   
unique        7043      2       2          2            2             3               3              3            3   
top     7590-VHVEG   Male      No         No          Yes           

In [27]:
# ── Dataset-Specific: Churn Rate Breakdown ───────────────────
print('='*60)
print('  CHURN RATE BREAKDOWN — Telco_CSV')
print('='*60)
churn_vc = telco_csv_df['Churn'].value_counts()
churn_pct = (churn_vc / len(telco_csv_df) * 100).round(2)
churn_summary = pd.DataFrame({'Count': churn_vc, 'Percentage': churn_pct})
print(churn_summary)

# ── SeniorCitizen Flag Breakdown ─────────────────────────────
print('\n  SeniorCitizen Distribution (0=No, 1=Yes):')
print(telco_csv_df['SeniorCitizen'].value_counts())

# ── Charges by Contract Type ─────────────────────────────────
print('\n  Monthly Charges by Contract Type:')
print(telco_csv_df.groupby('Contract')['MonthlyCharges']
      .agg(['mean', 'median', 'min', 'max']).round(2))

  CHURN RATE BREAKDOWN — Telco_CSV
       Count  Percentage
Churn                   
No      5174     73.4600
Yes     1869     26.5400

  SeniorCitizen Distribution (0=No, 1=Yes):
SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64

  Monthly Charges by Contract Type:
                  mean  median     min      max
Contract                                       
Month-to-month 66.4000 73.2500 18.7500 117.4500
One year       65.0500 68.7500 18.2500 118.6000
Two year       60.7700 64.3500 18.4000 118.7500


## Phase 5 — Dataset 3: Telco Churn Extended (Excel)

In [28]:
# ── Load ─────────────────────────────────────────────────────
telco_xl_df = pd.read_excel(r'D:\telecom-customer-analytics-platform\data\raw\Telco_customer_churn.xlsx',
                            sheet_name='Telco_Churn')
print('Excel loaded. Shape:', telco_xl_df.shape)
telco_xl_df.head()

Excel loaded. Shape: (7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.9641,-118.2728,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.8500,108.1500,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.0593,-118.3074,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7000,151.6500,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.0480,-118.2940,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.6500,820.5000,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.0621,-118.3157,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.8000,3046.0500,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.0392,-118.2663,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.7000,5036.3000,Yes,1,89,5340,Competitor had better devices


In [30]:
telco_xl_df[telco_xl_num_cols].dtypes

Latitude           float64
Longitude          float64
Tenure Months        int64
Monthly Charges    float64
Total Charges       object
Churn Value          int64
Churn Score          int64
CLTV                 int64
dtype: object

In [31]:
print("Blank Total Charges:")
print(
    telco_xl_df["Total Charges"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

Blank Total Charges:
11


In [32]:
telco_xl_df["Total Charges"] = (
    pd.to_numeric(
        telco_xl_df["Total Charges"],
        errors="coerce"
    )
)

In [33]:
print(telco_xl_df["Total Charges"].dtype)

float64


In [34]:
# ── Dataset-Specific Notes ────────────────────────────────────
# • Extended IBM Telco (7043 rows, 33 columns)
# • Adds: Churn Score, CLTV, Churn Reason, geographic columns
# • 'Lat Long' is a combined string — not usable as numeric directly
# • Some columns overlap with WA_Fn CSV (different naming convention)

# Drop 'Lat Long' combined string column before numeric analysis
# (Latitude and Longitude are already separate)
telco_xl_num_cols = [
    'Latitude', 'Longitude', 'Tenure Months',
    'Monthly Charges', 'Total Charges',
    'Churn Value', 'Churn Score', 'CLTV'
]

telco_xl_result = run_full_profile(
    df                 = telco_xl_df,
    name               = 'Telco_Excel',
    id_col             = 'CustomerID',
    force_numeric_cols = telco_xl_num_cols,
    dataset_color      = PALETTE[3],      # purple
    corr_cmap          = 'Purples'
)


██████████████████████████████████████████████████████████████████████
  PROFILING START : Telco_Excel
██████████████████████████████████████████████████████████████████████

  DATASET OVERVIEW — Telco_Excel
  Rows               : 7,043
  Columns            : 33
  Memory Usage (MB)  : 10.3374

  COLUMN NAMES — Telco_Excel
    1. CustomerID
    2. Count
    3. Country
    4. State
    5. City
    6. Zip Code
    7. Lat Long
    8. Latitude
    9. Longitude
   10. Gender
   11. Senior Citizen
   12. Partner
   13. Dependents
   14. Tenure Months
   15. Phone Service
   16. Multiple Lines
   17. Internet Service
   18. Online Security
   19. Online Backup
   20. Device Protection
   21. Tech Support
   22. Streaming TV
   23. Streaming Movies
   24. Contract
   25. Paperless Billing
   26. Payment Method
   27. Monthly Charges
   28. Total Charges
   29. Churn Label
   30. Churn Value
   31. Churn Score
   32. CLTV
   33. Churn Reason

  DATA TYPES — Telco_Excel
           Column   DType

In [35]:
# ── Dataset-Specific: CLTV Segmentation ─────────────────────
print('='*60)
print('  CLTV DISTRIBUTION — Telco_Excel')
print('='*60)
telco_xl_df['CLTV_Segment'] = pd.qcut(
    telco_xl_df['CLTV'], q=4,
    labels=['Bronze', 'Silver', 'Gold', 'Platinum']
)
print(telco_xl_df['CLTV_Segment'].value_counts())

# ── Churn Reason Top 10 ──────────────────────────────────────
print('\n  Top 10 Churn Reasons:')
print(telco_xl_df['Churn Reason'].value_counts().head(10))

# ── Churn Score Stats ────────────────────────────────────────
print('\n  Churn Score Stats:')
print(telco_xl_df['Churn Score'].describe().round(2))

  CLTV DISTRIBUTION — Telco_Excel
CLTV_Segment
Bronze      1763
Platinum    1761
Gold        1760
Silver      1759
Name: count, dtype: int64

  Top 10 Churn Reasons:
Churn Reason
Attitude of support person                   192
Competitor offered higher download speeds    189
Competitor offered more data                 162
Don't know                                   154
Competitor made better offer                 140
Attitude of service provider                 135
Competitor had better devices                130
Network reliability                          103
Product dissatisfaction                      102
Price too high                                98
Name: count, dtype: int64

  Churn Score Stats:
count   7043.0000
mean      58.7000
std       21.5300
min        5.0000
25%       40.0000
50%       61.0000
75%       75.0000
max      100.0000
Name: Churn Score, dtype: float64


## Phase 6 — Dataset 4: OpenCelliD Cell Towers (410.csv)

In [36]:
# ── Load (no header — OpenCelliD format) ─────────────────────
CELLTOWER_COLS = [
    'radio', 'mcc', 'net', 'area', 'cell', 'unit',
    'lon', 'lat', 'range', 'samples', 'changeable',
    'created', 'updated', 'averageSignal'
]
cell_df = pd.read_csv(r'D:\telecom-customer-analytics-platform\data\raw\410.csv',
                      header=None, names=CELLTOWER_COLS)
print('410.csv (Cell Towers) loaded. Shape:', cell_df.shape)
cell_df.head()

410.csv (Cell Towers) loaded. Shape: (4225, 14)


,radio,mcc,net,area,cell,unit,lon,lat,range,samples,changeable,created,updated,averageSignal
0,GSM,410,3,52203,32307,0,71.5061,33.5514,1000,8,1,1387347609,1763025241,0
1,GSM,410,6,443,3433,0,71.4251,33.9551,1452,11,1,1399784437,1749770281,0
2,GSM,410,3,52204,32133,0,71.4274,33.9509,1000,10,1,1420720710,1750628043,0
3,GSM,410,6,280,62313,0,68.3961,28.2884,3523,6,1,1445849090,1753434062,0
4,LTE,410,4,32,11011,0,67.1172,24.9297,1000,3,1,1446762871,1748984582,0


In [37]:
# ── Dataset-Specific Notes ────────────────────────────────────
# • OpenCelliD format — no header row in file
# • mcc=410 → Pakistan; 'radio' is GSM/LTE/UMTS/NR
# • 'created' / 'updated' are Unix timestamps
# • 'averageSignal' is typically 0 (not reported) for most entries
# • 'cell' is the cell tower ID — high cardinality
# • Outlier detection meaningful for: range, samples, lat, lon

# Convert Unix timestamps for readability
cell_df['created_dt'] = pd.to_datetime(cell_df['created'], unit='s', errors='coerce')
cell_df['updated_dt'] = pd.to_datetime(cell_df['updated'], unit='s', errors='coerce')
print('Timestamp columns added.')

cell_result = run_full_profile(
    df                 = cell_df,
    name               = 'CellTowers_410',
    id_col             = None,   # composite key: mcc+net+area+cell
    force_numeric_cols = ['lat', 'lon', 'range', 'samples', 'averageSignal'],
    dataset_color      = PALETTE[4],   # orange
    corr_cmap          = 'Oranges'
)

Timestamp columns added.

██████████████████████████████████████████████████████████████████████
  PROFILING START : CellTowers_410
██████████████████████████████████████████████████████████████████████

  DATASET OVERVIEW — CellTowers_410
  Rows               : 4,225
  Columns            : 16
  Memory Usage (MB)  : 0.7258

  COLUMN NAMES — CellTowers_410
    1. radio
    2. mcc
    3. net
    4. area
    5. cell
    6. unit
    7. lon
    8. lat
    9. range
   10. samples
   11. changeable
   12. created
   13. updated
   14. averageSignal
   15. created_dt
   16. updated_dt

  DATA TYPES — CellTowers_410
       Column          DType
        radio         object
          mcc          int64
          net          int64
         area          int64
         cell          int64
         unit          int64
          lon        float64
          lat        float64
        range          int64
      samples          int64
   changeable          int64
      created          int64
      up

  Boxplot saved → profiling_outputs\plots\CellTowers_410_boxplots.png

  FREQUENCY DISTRIBUTION — CellTowers_410

  [radio]
radio
LTE     3409
UMTS     415
GSM      401
  Bar chart saved → profiling_outputs\plots\CellTowers_410_freq_radio.png

  CORRELATION MATRIX — CellTowers_410
               mcc     net    area    cell    unit     lon     lat   range  samples  changeable  created  updated  averageSignal
mcc            NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN      NaN         NaN      NaN      NaN            NaN
net            NaN  1.0000 -0.1990  0.1040  0.0160 -0.1140 -0.0620  0.0240   0.0720         NaN   0.0130  -0.0310            NaN
area           NaN -0.1990  1.0000 -0.2170 -0.0610  0.1830  0.1100 -0.0110  -0.0880         NaN   0.0930  -0.2060            NaN
cell           NaN  0.1040 -0.2170  1.0000 -0.1230  0.1270  0.1370 -0.0600   0.0410         NaN   0.2080   0.2240            NaN
unit           NaN  0.0160 -0.0610 -0.1230  1.0000 -0.1290 -0.1440  0.031

In [38]:
# ── Dataset-Specific: Radio Technology Breakdown ─────────────
print('='*60)
print('  RADIO TECHNOLOGY BREAKDOWN — CellTowers_410')
print('='*60)
radio_vc = cell_df['radio'].value_counts()
print(radio_vc)

# ── Network Operator (MNC) Distribution ──────────────────────
print('\n  MNC (Network Operator) Distribution:')
print(cell_df['net'].value_counts())

# ── AverageSignal filled vs zero ─────────────────────────────
sig_zero = (cell_df['averageSignal'] == 0).sum()
print(f'\n  averageSignal = 0 (unreported) : {sig_zero} / {len(cell_df)}  ({sig_zero/len(cell_df)*100:.1f}%)')

# ── Composite Key Duplicate Check ────────────────────────────
dup_key = cell_df.duplicated(subset=['mcc','net','area','cell']).sum()
print(f'\n  Composite Key (mcc+net+area+cell) Duplicates : {dup_key}')

  RADIO TECHNOLOGY BREAKDOWN — CellTowers_410
radio
LTE     3409
UMTS     415
GSM      401
Name: count, dtype: int64

  MNC (Network Operator) Distribution:
net
1     1300
4     1037
3      866
6      724
7      278
90       9
99       7
92       2
5        1
11       1
Name: count, dtype: int64

  averageSignal = 0 (unreported) : 4225 / 4225  (100.0%)

  Composite Key (mcc+net+area+cell) Duplicates : 5


## Phase 7 — Cross-Dataset Summary

In [39]:
# ============================================================
# CROSS-DATASET SUMMARY TABLE
# ============================================================

summary_rows = []
for ds_name, res in MASTER_RESULTS.items():
    ov  = res['overview']
    dup = res['duplicates']
    mv  = res['missing']
    total_missing = int(mv['Missing_Count'].sum())
    summary_rows.append({
        'Dataset':          ds_name,
        'Rows':             ov['rows'],
        'Columns':          ov['columns'],
        'Memory_MB':        ov['memory_mb'],
        'Duplicate_Rows':   dup['total_duplicate_rows'],
        'Duplicate_Pct':    dup['duplicate_pct'],
        'Total_Missing':    total_missing,
        'Cols_w_Missing':   int((mv['Missing_Count'] > 0).sum())
    })

cross_df = pd.DataFrame(summary_rows)
print('='*80)
print('  CROSS-DATASET SUMMARY')
print('='*80)
print(cross_df.to_string(index=False))

# Save to CSV
cross_df.to_csv(RESULTS_DIR / 'cross_dataset_summary.csv', index=False)
print('\n  Summary saved to results/cross_dataset_summary.csv')

  CROSS-DATASET SUMMARY
       Dataset  Rows  Columns  Memory_MB  Duplicate_Rows  Duplicate_Pct  Total_Missing  Cols_w_Missing
     PK_Cities   146        9     0.0489               0         0.0000            148               3
     Telco_CSV  7043       21     7.4196               0         0.0000             11               1
   Telco_Excel  7043       33    10.3374               0         0.0000           5185               2
CellTowers_410  4225       16     0.7258               0         0.0000              0               0

  Summary saved to results/cross_dataset_summary.csv


In [40]:
# ── Comparison Bar Chart ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ['Rows', 'Duplicate_Rows', 'Total_Missing']
titles  = ['Row Count', 'Duplicate Rows', 'Total Missing Values']
colors  = [PALETTE[0], PALETTE[1], PALETTE[2]]

for ax, metric, title, clr in zip(axes, metrics, titles, colors):
    ax.bar(cross_df['Dataset'], cross_df[metric], color=clr, alpha=0.85)
    ax.set_title(title, fontsize=10)
    ax.set_xticklabels(cross_df['Dataset'], rotation=15, ha='right', fontsize=8)
    ax.set_ylabel('Count')

fig.suptitle('Cross-Dataset Comparison', fontsize=13, y=1.02)
plt.tight_layout()
path = PLOTS_DIR / 'cross_dataset_comparison.png'
plt.savefig(path, dpi=120, bbox_inches='tight')
plt.close()
print(f'Comparison chart saved → {path}')

Comparison chart saved → profiling_outputs\plots\cross_dataset_comparison.png


In [41]:
print('\nAll profiling complete. Plots and results saved to:', OUTPUT_DIR.resolve())
print('\nFiles generated:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        print(' ', f)


All profiling complete. Plots and results saved to: D:\telecom-customer-analytics-platform\notebooks\profiling_outputs

Files generated:
  profiling_outputs\plots\CellTowers_410_boxplots.png
  profiling_outputs\plots\CellTowers_410_correlation.png
  profiling_outputs\plots\CellTowers_410_freq_radio.png
  profiling_outputs\plots\cross_dataset_comparison.png
  profiling_outputs\plots\PK_Cities_boxplots.png
  profiling_outputs\plots\PK_Cities_correlation.png
  profiling_outputs\plots\PK_Cities_freq_country.png
  profiling_outputs\plots\Telco_CSV_boxplots.png
  profiling_outputs\plots\Telco_CSV_correlation.png
  profiling_outputs\plots\Telco_CSV_freq_gender.png
  profiling_outputs\plots\Telco_Excel_boxplots.png
  profiling_outputs\plots\Telco_Excel_correlation.png
  profiling_outputs\plots\Telco_Excel_freq_Country.png
  profiling_outputs\results\cross_dataset_summary.csv
